![](automated ticket pipeline.jpg)

####1. Automated Support Triage & Ticket Routing
Customer service teams are often overwhelmed with a massive backlog of tickets. Instead of having humans read every ticket to decide who handles it, your pipeline does it instantly.

####NLP Techniques (Syntax & Structure)
- Tokenization: Splits text into individual words and punctuation.
- POS Tagging: Assigns grammatical labels (nouns, verbs, adjectives).
- Vector Embeddings: Converts text meaning into mathematical numbers.

####NLU Techniques (Meaning & Context)
- Named Entity Recognition (NER): Extracts specific real-world subjects (products, locations).
- Sentiment Analysis: Identifies emotional tone (Positive, Negative, Neutral).
- Intent Detection: Determines the user's specific goal (returns, complaints).

####How to create Databricks Token
- In your Databricks workspace, click your username in the top bar and select Settings.
- Click Developer.
- Next to Access tokens, click Manage.
- Click Generate new token.

In [0]:
#databricks secrets create-scope izgenaiscope
#databricks secrets put-secret izgenaiscope databricks_token
DATABRICKS_TOKEN = dbutils.secrets.get(scope="genaiscope", key="databricks_ai_token")

1. We are getting data into the Bronze layer of our Medallion Arch.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS lakehousecat;
CREATE SCHEMA IF NOT EXISTS lakehousecat.default;
drop table if exists lakehousecat.default.customer_reviews;
CREATE TABLE IF NOT EXISTS lakehousecat.default.customer_reviews (
  review_id STRING,
  city string,
  review_text STRING);
INSERT INTO lakehousecat.default.customer_reviews VALUES
("REV-001",'NYC', "The shoe are beautiful, but they ripped after two days of wearing them! I want my money back."),
("REV-002",'NY', "The Laptop Shipping took 3 weeks. Absolutely unacceptable."),
("REV-003",'CA', "Perfect fit! Will definitely be buying from you guys again."),
("REV-004",'CAL', "I received the wrong color. I ordered black but got blue. How do I fix this?");

INSERT INTO lakehousecat.default.customer_reviews VALUES
("REV-005",'NYC', "Do you know how much mark my son is going to get in this exam?");

INSERT INTO lakehousecat.default.customer_reviews VALUES
("REV-006",'NYC', "phone purchased a week ago for \$500 in NewYork");

In [0]:
df_reviews = spark.read.table("lakehousecat.default.customer_reviews")
display(df_reviews)

####NLP & NLU using Text Generation Model

here ai_query is a function.. which takes model name and prompt as i/p and return the output in a string? Is it a spark function or databricks func?

In [0]:
%sql
CREATE OR REPLACE VIEW lakehousecat.default.advanced_text_analysis AS
SELECT 
  review_id,
  review_text,

  -- NLP CONCEPTS: Syntax, Structure, & Mechanics
--Prompt engineering techniques we used here - Major 2 types of prompts - System & User Prompt (Role Prompt, Instruction Prompt, Output Prompt.)
  -- 1. Tokenization: Breaking text into individual pieces
  ai_query(
    'databricks-meta-llama-3-1-8b-instruct',
    CONCAT('You are a tokenizer. Break the following text down into individual words and punctuation marks. Return ONLY a semi-colon separated list of tokens: ', review_text)
  ) AS nlp_tokens,

  -- 2. POS (Part-of-Speech) Tagging: Identifying grammatical labels
  ai_query(
    'databricks-meta-llama-3-1-8b-instruct',
    CONCAT('You are a POS tagger. Identify the part of speech for the words in this text. Return ONLY a comma-separated list in "word(TAG)" format (e.g., fast(ADJECTIVE), run(VERB)): ', review_text)
  ) AS nlp_pos_tags,

  -- 3. Vector Embeddings: Converting tex into mathematical (number) representations
  -- Note: Text generation models (like Llama) don't create embeddings. 
  ai_query(
    'databricks-bge-large-en', 
    review_text) AS nlp_vector_embedded_values,

--The below rephrased words is not the part of NLP operation, just to show GenAI generates rather than use the existing code.
  ai_query(
    'databricks-meta-llama-3-1-8b-instruct',
    CONCAT('You are a rephraser. From the given text, can you rephrase the entire text in 2 other ways. Return rephrased sentence in double quotes terminated by comma: ', review_text)
  ) AS example_of_GenAI_rephrased_words,

  -- NLU CONCEPTS: Meaning, Intent, Sentiment & Context
  --NLU 2 major roles - 1. Intent Identification , 2. Entity Recognition.

  -- 4. NER (Named Entity Recognition): Extracting specific real-world subjects
  ai_query(
    'databricks-meta-llama-3-1-8b-instruct',
    CONCAT('You are an NER tool., Extract Named Entities such as Person, Organization, Duration, Time, Money, Location, or Product. Return ONLY a comma-separated list in "Entity (Type)" format. If none exist, return "None": ', review_text)
  ) AS nlu_ner_entities,

  -- 5. Semantic Matching (Similarity Scoring) : Checking for a specific underlying goal (This example is to show the similarity scoring used behind)
  --by converting intent & given prompt into vectors
  --product is not useful,money back [1.2,4.1]= refund [1.2,4]
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT('Does this review express an intent to return the product, ask for a refund, or complain about shipping duration? Return ONLY the word YES or NO: ', review_text)
  ) AS nlu_return_semantic_matching,

  -- 6. Intent Classification/Generation: Identifying the broad purpose of the user
  ai_query(
    'databricks-meta-llama-3-1-8b-instruct',
    CONCAT('Identify the primary intent of this customer review. Choose ONLY ONE from the following categories: "Information", "Product Inquiry", "Complaint", "Praise", "Feature Request", "Customer Support", or "Other". Return ONLY the category name: ', review_text)
  ) AS nlu_primary_8b_intent,

  -- 6. Intent Classification/Generation: Identifying the broad purpose of the user
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT('Identify the primary intent of this customer review. Choose ONLY ONE from the following categories: "Information", "Product Inquiry", "Complaint", "Praise", "Feature Request", "Customer Support", or "Other". Return ONLY the category name: ', review_text)
  ) AS nlu_primary_70b_intent,
  
  -- 7. Intent Classification/Generation: Identifying the broad purpose of the user
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT('Identify the primary intent & sentiment of this customer review. Choose ONLY ONE from the following department we have to route this intented and sentiment to customer request such as: "Logistics Department", "Support Department","Quality Control Department", "Escalation" or "Other". Return ONLY the category name: ', review_text)
  ) AS nlu_primary_70b_intent_routing,

  -- 7. Sentiment Analysis: Understanding emotional tone
  ai_query(
    'databricks-meta-llama-3-1-8b-instruct',
    CONCAT('You are a sentiment analyzer. Classify the emotional tone of this review, dont hallucinate. Return sentiment score in a range of -1 to 1: ', review_text)
  ) AS sentiment_scoring

FROM lakehousecat.default.customer_reviews;

SELECT * FROM lakehousecat.default.advanced_text_analysis;

In [0]:
%sql
select uniqueid,ai_query(
    'databricks-meta-llama-3-1-8b-instruct',concat('you are a doctor, you have to suggest a treatment for the following condition',condition)) as treatment
     from catalog1_we4.schema1_we4.drug_info

![](automated mail generator.jpg)

####2. Automated Customer Review Response Generator (E-Commerce)
Instead of just classifying the sentiments and intents, We can use LLM (NLG) to actually write the email response to the customer.

The Pipeline Concept: Read the negative review from a Delta table, pass it to the LLM with strict instructions, and output a drafted email into a new column.

LLM & GenAI: LLM is a Large Lang Model using Meta Llama LLM, we are going to generate the mail output using NLG (GenAI+LLM) based on the input review text.

NLG (Natural Language Generation): The model synthesizes a polite, grammatically perfect, and empathetic email (as per our prompt)

Grounding: Pass the company's official return policy into the system prompt. The AI is grounded in this specific document so it doesn't make up rules.

Guardrailing: Add a strict rule to the prompt: "Never promise a refund or discount. Only offer to connect them to the support team."

Hallucination (Anti) Mitigation: Set temperature=0.1, so the AI doesn't invent fake product features or invent a fake customer service rep name.

In [0]:
%sql
CREATE OR REPLACE TABLE lakehousecat.default.ecommerce_raw_reviews (
  review_id STRING,
  customer_review STRING,
  custname string
);

INSERT INTO lakehousecat.default.ecommerce_raw_reviews (review_id, customer_review,custname)
VALUES 
  ('REV-001', 'The shoes are beautiful, but they ripped after two days of wearing them! I want my money back.','Irfan'),
  ('REV-002', 'Shipping took 3 weeks. Absolutely unacceptable.','Vaanmathy'),
  ('REV-003', 'Perfect fit! Will definitely be buying from you guys again.','Sarangabani'),
  ('REV-004', 'I received the wrong color. I ordered black but got blue. How do I fix this?','Vasu');

SELECT * FROM lakehousecat.default.ecommerce_raw_reviews;

In [0]:
%sql
--What we learn here - Prompt Engineering, Context Engineering, Grounding, Guardrailing, Anti Hallucination, Human In Loop, Model Parameters, NLG
SELECT 
    review_id,
    customer_review,
    custname,
    ai_query(
        -- 1. The Model Endpoint
        'databricks-meta-llama-3-3-70b-instruct',
        
        -- 2. The Contextual Prompt (System prompt/context (Grounding/Guardrailing/Anti Hallucination) + The Data Column (User Prompt))
        CONCAT(
            'You are an empathetic, professional customer support Engineer for an E-Commerce company. ',
            'Read the customer review and write a direct email response to them addressing the customer. ',
            'Produce the output with subject, salutation, body, closing message, and signature with - Thanks & Regards, Inceptez Technologies',
            
            'GROUNDING CONTEXT: ',
            '- We only accept returns within 30 days of purchase. ',
            '- We do NOT give cash refunds. We only offer store credit or exact item replacements. ',
            '- We offer discounts to the customers who have purchased once and gave positive review.',
            
            'GUARDRAILS: ',
            '1. Never promise a refund through electronic money transfer under any circumstances. ',
            '2. Never offer a discount code or coupon. ',
            '3. Keep the response under 4 sentences when you do NLG. ',
            '4. Never offer store credit or item replacements if the day of purchase is more than 1 day. ',
            
            'ANTI HALLUCINATION: ',
            'If the user asks for something not covered in the policies above, do not invent a solution. ',
            'Simply state: "I am escalating this to our senior support team who will contact you within 24 hours." ',
            'Redirect the customre to support team, for offering discount code or coupon to the customers who have purchased once and gave positive review. ',
            
            'Customer Review: ', customer_review, custname
        ),
        
        -- 3. The Model Parameters (Ensuring robotic/strict adherence to guardrails)
        modelParameters => named_struct('temperature', 0, 'max_tokens', 100)
        
    ) AS ai_generated_email

FROM lakehousecat.default.ecommerce_raw_reviews;